# Modul 04: Explorative Datenanalyse und manuelle Vorverarbeitung | Lösungen

## Überblick

Sie erkunden einen absichtlich unordentlichen Kundendatensatz, visualisieren Verteilungen und Beziehungen und entwickeln anschließend eine nachvollziehbare manuelle Vorverarbeitung. Entscheidungen zu Fehlwerten, Ausreißern, Einheiten, Kategorien und Transformationen werden ausdrücklich kontrolliert.

**Zugehörige Vorlesungen**

- **Daten erkunden**
- **Manuell vorverarbeiten**

## Lernziele

Nach der Bearbeitung können Sie:

- Tabellen mit Kennzahlen, Häufigkeiten und geeigneten Diagrammen explorativ untersuchen.
- fehlende, doppelte, unplausible und uneinheitliche Werte fachlich begründet bereinigen.
- numerische, kategoriale, Text-, Einheiten- und Datumsmerkmale reproduzierbar transformieren und kontrollieren.

## Geprüfte Fähigkeiten

- EDA mit describe, value_counts, Gruppenkennzahlen, Histogramm, Boxplot und Streudiagramm
- manuelle Imputation, Typkonvertierung, Kategorie- und Einheitennormalisierung
- Ausreißermarkierung, Skalierung, Binning, Log-Transformation und wiederverwendbare Prüfungen

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** mittel
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Der folgende DataFrame enthält absichtlich Fehlwerte, Duplikate, gemischte Einheiten, Schreibvarianten, uneinheitliche Datumsformate und einen extremen Kaufbetrag.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42

kunden_roh = pd.DataFrame(
    {
        "kunde_id": [101, 102, 103, 104, 105, 106, 106, 107, 108, 109, 110, 111],
        "alter": [29, "34", 41, np.nan, 52, 38, 38, "unbekannt", 47, 31, 63, 44],
        "einkommen_eur": [42000, 51000, np.nan, 47000, 68000, 55000, 55000, 39000, 72000, 46000, 81000, np.nan],
        "stadt": ["Berlin", " berlin ", "MÜNCHEN", "München", "Hamburg", "hamburg", "hamburg", "Berlin", "Köln", "koeln", "Berlin", "München"],
        "gewicht": ["72 kg", "158 lb", "81kg", "70 KG", np.nan, "176 lb", "176 lb", "65kg", "90 kg", "143 lb", "84kg", "75 kg"],
        "anmeldung": ["2026-01-05", "05.02.2026", "2026/03/12", "2026-04-01", "15.05.2026", "2026-06-20", "2026-06-20", "2026-07-02", "2026-07-14", "2026-07-21", "2026-08-01", "2026-08-09"],
        "kaeufe": [3, 5, 4, 6, 8, 5, 5, 2, 7, 4, 40, 6],
        "umsatz_eur": [320, 510, 410, 620, 980, 560, 560, 210, 880, 430, 9500, 690],
    }
)

print("Einrichtung abgeschlossen.")
print("Rohdatenform:", kunden_roh.shape)
display(kunden_roh)

### Aufgabe 1: Schneller EDA-Überblick

Erstellen Sie einen strukturierten ersten Überblick:

1. Form, Spaltennamen und Datentypen.
2. Fehlwerte je Spalte.
3. Anzahl vollständig doppelter Zeilen und doppelte `kunde_id`.
4. Numerische Kennzahlen für `kaeufe` und `umsatz_eur`, einschließlich Median.
5. Häufigkeiten der Rohwerte in `stadt`, sowohl absolut als auch relativ.
6. Mindestens drei auffällige Beobachtungen oder Fragen.

In [ ]:
eda_daten = kunden_roh.copy()

# ============================================================
# MUSTERLÖSUNG
# ============================================================

print("Form:", eda_daten.shape)
print("Spalten:", eda_daten.columns.tolist())
print("\nDatentypen:")
print(eda_daten.dtypes)

print("\nFehlwerte:")
print(eda_daten.isna().sum())

print("\nVollständige Duplikate:", int(eda_daten.duplicated().sum()))
print("Zusätzliche Vorkommen doppelter IDs:", int(eda_daten.duplicated("kunde_id").sum()))

# describe() liefert Standardkennzahlen; der Median wird explizit ergänzt.
numerische_uebersicht = eda_daten[["kaeufe", "umsatz_eur"]].describe().T
numerische_uebersicht["median"] = eda_daten[["kaeufe", "umsatz_eur"]].median()
display(numerische_uebersicht)

stadt_absolut = eda_daten["stadt"].value_counts(dropna=False)
stadt_relativ = eda_daten["stadt"].value_counts(dropna=False, normalize=True)
stadt_bericht = pd.DataFrame({"Anzahl": stadt_absolut, "Anteil": stadt_relativ})
display(stadt_bericht)

print("Auffälligkeiten:")
print("1. Stadtwerte unterscheiden sich nur durch Großschreibung, Leerzeichen oder Umlautschreibweise.")
print("2. Alter und Gewicht sind als gemischte Objekttypen gespeichert.")
print("3. Kunde 106 ist doppelt enthalten.")
print("4. 40 Käufe und 9.500 EUR könnten echte Spitzenwerte oder Fehler sein.")

> **Musterantwort und Interpretation**
>
> Zuerst sollte geklärt werden, ob die doppelte ID 106 dieselbe Beobachtung oder zwei legitime Ereignisse darstellt. Diese Entscheidung beeinflusst Zeilenzahl, Kennzahlen und mögliche spätere Splits. Danach sind die extremen Kauf- und Umsatzwerte fachlich zu prüfen, statt sie automatisch zu löschen.

### Aufgabe 2: Verteilungen, Gruppen und Beziehungen visualisieren

Erstellen Sie eine Abbildung mit:

1. Histogramm des Umsatzes,
2. Boxplot der Käufe,
3. Streudiagramm von Käufen und Umsatz,
4. Balkendiagramm des mittleren Umsatzes je roher Stadtbezeichnung.

Berechnen Sie außerdem die Korrelation zwischen Käufen und Umsatz einmal mit allen Zeilen und einmal ohne die Zeile mit dem höchsten Umsatz.

In [ ]:
plot_roh = kunden_roh.copy()

# ============================================================
# MUSTERLÖSUNG
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(plot_roh["umsatz_eur"].dropna(), bins=7, edgecolor="black")
axes[0, 0].set_title("Umsatzverteilung mit Extremwert")
axes[0, 0].set_xlabel("Umsatz in EUR")
axes[0, 0].set_ylabel("Häufigkeit")

axes[0, 1].boxplot(plot_roh["kaeufe"].dropna(), vert=True)
axes[0, 1].set_title("Boxplot der Käufe")
axes[0, 1].set_ylabel("Anzahl Käufe")

axes[1, 0].scatter(plot_roh["kaeufe"], plot_roh["umsatz_eur"], alpha=0.8)
axes[1, 0].set_title("Käufe und Umsatz")
axes[1, 0].set_xlabel("Anzahl Käufe")
axes[1, 0].set_ylabel("Umsatz in EUR")

stadt_mittel = plot_roh.groupby("stadt")["umsatz_eur"].mean().sort_values()
axes[1, 1].barh(stadt_mittel.index, stadt_mittel.values)
axes[1, 1].set_title("Mittlerer Umsatz je roher Stadtbezeichnung")
axes[1, 1].set_xlabel("Mittlerer Umsatz in EUR")
axes[1, 1].set_ylabel("Rohwert Stadt")

plt.tight_layout()
plt.show()

# Der Extremwert kann eine lineare Korrelation stark beeinflussen.
korrelation_alle = plot_roh[["kaeufe", "umsatz_eur"]].corr().loc["kaeufe", "umsatz_eur"]
ohne_maximum = plot_roh.drop(index=plot_roh["umsatz_eur"].idxmax())
korrelation_ohne_max = ohne_maximum[["kaeufe", "umsatz_eur"]].corr().loc["kaeufe", "umsatz_eur"]

print(f"Korrelation mit allen Zeilen: {korrelation_alle:.3f}")
print(f"Korrelation ohne höchsten Umsatz: {korrelation_ohne_max:.3f}")

> **Musterantwort und Interpretation**
>
> Der Vergleich zeigt, wie stark ein einzelner Extremwert die gemessene lineare Beziehung verändern kann. Selbst eine hohe Korrelation beweist nicht, dass mehr Käufe den Umsatz unabhängig von Preisen, Kundentyp oder Zeitraum verursachen. Beide Größen können durch weitere Faktoren gemeinsam beeinflusst werden.

### Aufgabe 3: Duplikate, Datentypen und Fehlwerte bereinigen

Erzeugen Sie `kunden_basis` mit folgenden Schritten:

1. Entfernen Sie nur vollständig identische Duplikate.
2. Wandeln Sie `alter` mit `pd.to_numeric(..., errors="coerce")` um.
3. Ersetzen Sie fehlendes Alter mit dem Median.
4. Ersetzen Sie fehlendes Einkommen mit dem Median der bereits bereinigten Stadtgruppe. Nutzen Sie vorerst eine einfache Stadt-Normalisierung.
5. Erzeugen Sie Indikatorspalten `alter_war_fehlend` und `einkommen_war_fehlend`, bevor Sie imputieren.
6. Kontrollieren Sie Formen und verbleibende Fehlwerte.

In [ ]:
kunden_basis = kunden_roh.copy()

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Vollständig identische Zeilen können entfernt werden, nachdem ihre Bedeutung geprüft wurde.
kunden_basis = kunden_basis.drop_duplicates().copy()

# Eine einfache Normalisierung dient hier nur zur gruppenspezifischen Imputation.
kunden_basis["stadt_norm"] = (
    kunden_basis["stadt"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"koeln": "köln"})
)

# Die Indikatoren bewahren die Information, dass ein Wert ursprünglich fehlte.
kunden_basis["alter_war_fehlend"] = kunden_basis["alter"].isna() | (
    pd.to_numeric(kunden_basis["alter"], errors="coerce").isna()
)
kunden_basis["einkommen_war_fehlend"] = kunden_basis["einkommen_eur"].isna()

# Nichtnumerische Alterswerte werden zu NaN und anschließend robust mit dem Median ersetzt.
kunden_basis["alter"] = pd.to_numeric(kunden_basis["alter"], errors="coerce")
alter_median = kunden_basis["alter"].median()
kunden_basis["alter"] = kunden_basis["alter"].fillna(alter_median)

# transform() erzeugt pro Zeile den Median ihrer normalisierten Stadtgruppe.
stadt_einkommensmedian = kunden_basis.groupby("stadt_norm")["einkommen_eur"].transform("median")
kunden_basis["einkommen_eur"] = kunden_basis["einkommen_eur"].fillna(stadt_einkommensmedian)

# Falls eine ganze Gruppe keinen Wert hätte, dient der globale Median als dokumentierter Fallback.
kunden_basis["einkommen_eur"] = kunden_basis["einkommen_eur"].fillna(
    kunden_basis["einkommen_eur"].median()
)

print("Form nach Duplikatentfernung:", kunden_basis.shape)
print("Verbleibende Fehlwerte:")
print(kunden_basis[["alter", "einkommen_eur"]].isna().sum())
display(kunden_basis.head())

> **Musterantwort und Interpretation**
>
> Nach der Ersetzung ist sonst nicht mehr erkennbar, welche Werte beobachtet und welche geschätzt wurden. Ein Indikator ermöglicht Qualitätskontrollen und kann später zeigen, ob das Fehlen selbst systematisch mit dem Ziel zusammenhängt.

### Aufgabe 4: Einheiten, Texte und Datumswerte vereinheitlichen

1. Schreiben Sie `parse_gewicht_kg(wert)`, das Angaben in kg oder lb in Kilogramm umwandelt. Verwenden Sie `1 lb = 0.453592 kg`.
2. Vereinheitlichen Sie Städte zu `Berlin`, `München`, `Hamburg` und `Köln`.
3. Konvertieren Sie `anmeldung` robust in ein Datumsformat. Gemischte Formate müssen berücksichtigt werden.
4. Erzeugen Sie `tage_seit_anmeldung` relativ zum Stichtag `2026-08-25`.
5. Prüfen Sie, ob Gewichte und Datumswerte plausibel sind.

In [ ]:
def parse_gewicht_kg(wert):
    """Wandelt eine Gewichtsangabe in Kilogramm um oder liefert np.nan."""
    pass


kunden_transformiert = kunden_basis.copy()
stichtag = pd.Timestamp("2026-08-25")

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def parse_gewicht_kg(wert):
    """Wandelt eine Gewichtsangabe in Kilogramm um oder liefert np.nan."""
    if pd.isna(wert):
        return np.nan

    # Leerzeichen und Großschreibung werden entfernt, damit Varianten gleich behandelt werden.
    text = str(wert).strip().lower().replace(" ", "")

    if text.endswith("kg"):
        zahl = text[:-2]
        faktor = 1.0
    elif text.endswith("lb"):
        zahl = text[:-2]
        faktor = 0.453592
    else:
        return np.nan

    try:
        return float(zahl) * faktor
    except ValueError:
        return np.nan


kunden_transformiert["gewicht_kg"] = kunden_transformiert["gewicht"].map(parse_gewicht_kg)

# Eine explizite Zuordnung erhält verständliche, einheitliche Kategorien.
stadt_mapping = {
    "berlin": "Berlin",
    "münchen": "München",
    "hamburg": "Hamburg",
    "köln": "Köln",
    "koeln": "Köln",
}
roh_normalisiert = kunden_transformiert["stadt"].astype("string").str.strip().str.lower()
kunden_transformiert["stadt_norm"] = roh_normalisiert.map(stadt_mapping)

# format="mixed" lässt pandas jedes Element separat interpretieren.
kunden_transformiert["anmeldung_dt"] = pd.to_datetime(
    kunden_transformiert["anmeldung"],
    format="mixed",
    dayfirst=True,
    errors="coerce",
)
kunden_transformiert["tage_seit_anmeldung"] = (
    stichtag - kunden_transformiert["anmeldung_dt"]
).dt.days

# Plausibilitätsprüfungen markieren, statt Daten still zu löschen.
kunden_transformiert["gewicht_plausibel"] = kunden_transformiert["gewicht_kg"].between(35, 250)
kunden_transformiert["datum_plausibel"] = (
    kunden_transformiert["anmeldung_dt"].notna()
    & (kunden_transformiert["anmeldung_dt"] <= stichtag)
)

display(
    kunden_transformiert[
        ["gewicht", "gewicht_kg", "stadt", "stadt_norm", "anmeldung_dt", "tage_seit_anmeldung"]
    ]
)
print("Nicht plausible Gewichte:", int((~kunden_transformiert["gewicht_plausibel"]).sum()))
print("Nicht plausible Datumswerte:", int((~kunden_transformiert["datum_plausibel"]).sum()))

### Aufgabe 5: Ausreißer markieren und Merkmale transformieren

1. Markieren Sie Umsatz-Ausreißer mit der IQR-Regel, ohne sie zu löschen.
2. Erzeugen Sie eine Min-Max-Skalierung von `einkommen_eur` manuell.
3. Teilen Sie Alter in nachvollziehbare Klassen, zum Beispiel `unter 35`, `35 bis 49`, `50 plus`.
4. Erzeugen Sie `log_umsatz = log1p(umsatz_eur)`.
5. Vergleichen Sie Rohumsatz und Log-Umsatz mit zwei Histogrammen.

In [ ]:
kunden_merkmale = kunden_transformiert.copy()

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Die IQR-Regel nutzt robuste Quartile und markiert Werte außerhalb von 1,5 IQR.
q1 = kunden_merkmale["umsatz_eur"].quantile(0.25)
q3 = kunden_merkmale["umsatz_eur"].quantile(0.75)
iqr = q3 - q1
untere_grenze = q1 - 1.5 * iqr
obere_grenze = q3 + 1.5 * iqr
kunden_merkmale["umsatz_ausreisser"] = ~kunden_merkmale["umsatz_eur"].between(
    untere_grenze, obere_grenze
)

# Min-Max-Skalierung wird hier bewusst manuell berechnet.
einkommen_min = kunden_merkmale["einkommen_eur"].min()
einkommen_max = kunden_merkmale["einkommen_eur"].max()
kunden_merkmale["einkommen_minmax"] = (
    kunden_merkmale["einkommen_eur"] - einkommen_min
) / (einkommen_max - einkommen_min)

# Binning übersetzt eine kontinuierliche Zahl in fachlich benannte Bereiche.
kunden_merkmale["altersklasse"] = pd.cut(
    kunden_merkmale["alter"],
    bins=[0, 35, 50, np.inf],
    right=False,
    labels=["unter 35", "35 bis 49", "50 plus"],
)

# log1p ist auch für den Wert 0 definiert und komprimiert große Abstände.
kunden_merkmale["log_umsatz"] = np.log1p(kunden_merkmale["umsatz_eur"])

print("IQR-Grenzen:", untere_grenze, obere_grenze)
print("Markierte Ausreißer:", int(kunden_merkmale["umsatz_ausreisser"].sum()))
print("Min-Max-Bereich:", kunden_merkmale["einkommen_minmax"].min(), kunden_merkmale["einkommen_minmax"].max())

display(kunden_merkmale[["alter", "altersklasse", "umsatz_eur", "umsatz_ausreisser"]])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(kunden_merkmale["umsatz_eur"], bins=7, edgecolor="black")
axes[0].set_title("Rohumsatz")
axes[0].set_xlabel("EUR")
axes[0].set_ylabel("Häufigkeit")

axes[1].hist(kunden_merkmale["log_umsatz"], bins=7, edgecolor="black")
axes[1].set_title("Log-transformierter Umsatz")
axes[1].set_xlabel("log(1 + Umsatz)")
axes[1].set_ylabel("Häufigkeit")
plt.tight_layout()
plt.show()

> **Musterantwort und Interpretation**
>
> Binning ersetzt exakte Alterswerte durch gröbere Gruppen und verliert Unterschiede innerhalb eines Intervalls. Es kann trotzdem sinnvoll sein, wenn fachliche Schwellen wichtig sind, die Darstellung verständlicher wird oder einfache nichtlineare Effekte abgebildet werden sollen.

### Aufgabe 6: Integrationsaufgabe: überprüfbare Vorverarbeitungsfunktion

Implementieren Sie `preprocess_kunden(input_daten)`. Die Funktion soll:

- auf einer Kopie arbeiten,
- vollständige Duplikate entfernen,
- Alter und Einkommen wie oben bereinigen,
- Stadt, Gewicht und Datum vereinheitlichen,
- Fehlwert- und Ausreißerindikatoren erzeugen,
- `tage_seit_anmeldung`, `altersklasse` und `log_umsatz` erzeugen,
- einen Kontrollbericht mit Zeilenzahl, Fehlwerten, eindeutigen IDs und markierten Ausreißern zurückgeben.

Die Funktion soll `(bereinigte_daten, kontrollbericht)` liefern und bei erneutem Aufruf auf den Rohdaten dasselbe Ergebnis erzeugen.

In [ ]:
def preprocess_kunden(input_daten):
    """Bereinigt Kundendaten und gibt Daten plus Kontrollbericht zurück."""
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def preprocess_kunden(input_daten):
    """Bereinigt Kundendaten und gibt Daten plus Kontrollbericht zurück."""
    daten = input_daten.copy()
    zeilen_vorher = len(daten)

    # Nur vollständig identische Zeilen werden ohne weitere Schlüsselannahme entfernt.
    daten = daten.drop_duplicates().copy()

    # Kategorien werden früh vereinheitlicht, damit gruppenbezogene Schritte reproduzierbar sind.
    stadt_mapping = {
        "berlin": "Berlin",
        "münchen": "München",
        "hamburg": "Hamburg",
        "köln": "Köln",
        "koeln": "Köln",
    }
    stadt_roh = daten["stadt"].astype("string").str.strip().str.lower()
    daten["stadt_norm"] = stadt_roh.map(stadt_mapping)

    # Fehlwertindikatoren müssen vor der Imputation entstehen.
    alter_numerisch = pd.to_numeric(daten["alter"], errors="coerce")
    daten["alter_war_fehlend"] = alter_numerisch.isna()
    daten["einkommen_war_fehlend"] = daten["einkommen_eur"].isna()

    daten["alter"] = alter_numerisch.fillna(alter_numerisch.median())

    gruppenmedian = daten.groupby("stadt_norm")["einkommen_eur"].transform("median")
    daten["einkommen_eur"] = daten["einkommen_eur"].fillna(gruppenmedian)
    daten["einkommen_eur"] = daten["einkommen_eur"].fillna(daten["einkommen_eur"].median())

    daten["gewicht_kg"] = daten["gewicht"].map(parse_gewicht_kg)
    daten["anmeldung_dt"] = pd.to_datetime(
        daten["anmeldung"], format="mixed", dayfirst=True, errors="coerce"
    )
    stichtag = pd.Timestamp("2026-08-25")
    daten["tage_seit_anmeldung"] = (stichtag - daten["anmeldung_dt"]).dt.days

    daten["altersklasse"] = pd.cut(
        daten["alter"],
        bins=[0, 35, 50, np.inf],
        right=False,
        labels=["unter 35", "35 bis 49", "50 plus"],
    )
    daten["log_umsatz"] = np.log1p(daten["umsatz_eur"])

    q1 = daten["umsatz_eur"].quantile(0.25)
    q3 = daten["umsatz_eur"].quantile(0.75)
    iqr = q3 - q1
    daten["umsatz_ausreisser"] = ~daten["umsatz_eur"].between(
        q1 - 1.5 * iqr, q3 + 1.5 * iqr
    )

    # Der Bericht macht zentrale Auswirkungen der Transformation sichtbar.
    kontrollbericht = pd.DataFrame(
        {
            "Prüfung": [
                "Zeilen vorher",
                "Zeilen nachher",
                "Doppelte IDs zusätzlich",
                "Fehlwerte in Kernmerkmalen",
                "Markierte Umsatz-Ausreißer",
            ],
            "Wert": [
                zeilen_vorher,
                len(daten),
                int(daten.duplicated("kunde_id").sum()),
                int(daten[["alter", "einkommen_eur", "stadt_norm", "gewicht_kg", "anmeldung_dt"]].isna().sum().sum()),
                int(daten["umsatz_ausreisser"].sum()),
            ],
        }
    )

    return daten, kontrollbericht


kunden_final, kontrollbericht = preprocess_kunden(kunden_roh)
display(kontrollbericht)
display(kunden_final.head())

# Determinismusprüfung: Derselbe Input muss dieselbe aufbereitete Tabelle liefern.
kunden_final_zwei, kontrollbericht_zwei = preprocess_kunden(kunden_roh)
pd.testing.assert_frame_equal(kunden_final, kunden_final_zwei)
pd.testing.assert_frame_equal(kontrollbericht, kontrollbericht_zwei)

> **Musterantwort und Interpretation**
>
> Nein. Die IQR-Regel liefert nur einen statistischen Hinweis. Der Wert kann ein echter Großkunde, eine andere Beobachtungseinheit oder ein Eingabefehler sein. Erst nach fachlicher Prüfung wird entschieden, ob er korrigiert, getrennt analysiert, robust modelliert oder ausgeschlossen wird. Die Entscheidung ist zu dokumentieren.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?